# 06 — Forest Models: Random Forest & XGBoost

Trains and evaluates Random Forest and XGBoost classifiers across the three feature spaces built in notebook 03.

The vault split is **locked** and not touched here — it is reserved for final evaluation in notebook 08.

## Setup

In [ ]:
import subprocess
subprocess.run(['pip', 'install', 'xgboost', '-q'], check=True)
print('XGBoost installed.')

In [ ]:
import sys
sys.path.append('..')

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

from config import TARGET_CLASSES, RANDOM_STATE

sns.set_theme(style='whitegrid')
plt.rcParams['figure.figsize'] = (9, 5)
os.makedirs('../figures/06-Forests', exist_ok=True)

## Load Data

In [ ]:
X_train      = joblib.load('../data/processed/X_train.pkl')
y_train      = joblib.load('../data/processed/y_train.pkl')
feature_sets = joblib.load('../data/processed/feature_sets.pkl')

print(f'Train : {X_train.shape}')
print(f'\nClass distribution (train):')
vc = pd.Series(y_train.values).value_counts().sort_index()
for idx, cnt in vc.items():
    print(f'  {idx} ({TARGET_CLASSES[idx]:<25}) — {cnt} samples')
print(f'\nFeature sets loaded: {list(feature_sets.keys())}')

## Class Balance Strategy

Both models handle class imbalance differently:
- **Random Forest** accepts `class_weight='balanced'` directly, which adjusts splits based on class frequency.
- **XGBoost** uses `sample_weight` passed at fit time via `compute_sample_weight('balanced')`.

In [ ]:
from sklearn.utils.class_weight import compute_sample_weight

preview_weights = compute_sample_weight('balanced', y_train)
print(f'Sample weight range: {preview_weights.min():.3f} → {preview_weights.max():.3f}')
print(f'(Higher = rarer class, lower = more common class)')

## Cross-Validation Setup

**Hierarchical CV scheme:**
- 20% of the total dataset is held out as the **vault** — never loaded here
- All model selection happens via **5-fold stratified CV** on the remaining 80%
- Transformers are **re-fit inside each fold** on that fold's training portion only — prevents data leakage
- Sample weights are recomputed fresh inside each fold

In [ ]:
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import accuracy_score, f1_score
from sklearn.base import clone

CV = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

def cv_evaluate(X_raw, y, model, transformer=None, use_sample_weight=False, verbose=False):
    """
    5-fold stratified CV.

    Parameters
    ----------
    X_raw            : raw feature array (pre-transformation)
    y                : labels
    model            : sklearn-compatible classifier
    transformer      : optional transformer re-fit per fold to prevent leakage
    use_sample_weight: pass balanced sample weights at fit time
    """
    y_arr = np.array(y)
    accs, f1s = [], []

    for fold, (tr_idx, val_idx) in enumerate(CV.split(X_raw, y_arr)):
        X_tr, X_val = X_raw[tr_idx], X_raw[val_idx]
        y_tr, y_val = y_arr[tr_idx], y_arr[val_idx]

        if transformer is not None:
            t = clone(transformer)
            X_tr  = t.fit_transform(X_tr, y_tr)
            X_val = t.transform(X_val)

        fit_kwargs = {}
        if use_sample_weight:
            fit_kwargs['sample_weight'] = compute_sample_weight('balanced', y_tr)

        m = clone(model)
        m.fit(X_tr, y_tr, **fit_kwargs)
        y_pred = m.predict(X_val)

        accs.append(accuracy_score(y_val, y_pred))
        f1s.append(f1_score(y_val, y_pred, average='macro'))

        if verbose:
            print(f'  Fold {fold+1}: acc={accs[-1]:.4f}  macro-F1={f1s[-1]:.4f}')

    return {
        'acc_mean': np.mean(accs), 'acc_std': np.std(accs),
        'f1_mean' : np.mean(f1s),  'f1_std' : np.std(f1s),
    }

print('CV helper ready — StratifiedKFold(n_splits=5)')

## Models

### Random Forest
An ensemble of decision trees trained on random subsets of features and data. `class_weight='balanced'` handles imbalance natively.

In [ ]:
from sklearn.ensemble import RandomForestClassifier

rf = RandomForestClassifier(
    n_estimators=200,
    class_weight='balanced',
    random_state=RANDOM_STATE,
    n_jobs=-1,
)

### XGBoost
Gradient boosted trees. Uses `sample_weight` at fit time for class balance since it does not accept `class_weight` directly.

In [ ]:
from xgboost import XGBClassifier

xgb = XGBClassifier(
    n_estimators=200,
    learning_rate=0.1,
    max_depth=6,
    use_label_encoder=False,
    eval_metric='mlogloss',
    random_state=RANDOM_STATE,
    n_jobs=-1,
)

## Run Cross-Validation

In [ ]:
from sklearn.decomposition import PCA
from sklearn.feature_selection import SelectFromModel
from sklearn.linear_model import LogisticRegression

cv_transformers = {
    'original': None,
    'pca'     : PCA(n_components=0.95, random_state=RANDOM_STATE),
    'lasso'   : SelectFromModel(
                    LogisticRegression(solver='saga', l1_ratio=1, C=0.1,
                                       max_iter=2000, random_state=RANDOM_STATE,
                                       class_weight='balanced'),
                    threshold=1e-6
                ),
}

models = {
    'Random Forest': (rf, False),   # class_weight handled internally
    'XGBoost'      : (xgb, True),   # needs sample_weight at fit time
}

cv_results = {}

for model_name, (model, use_sw) in models.items():
    cv_results[model_name] = {}
    for fs_name, fs in feature_sets.items():
        print(f'\n── {model_name} on \'{fs_name}\' ({fs["n_features"]} features) ──')
        scores = cv_evaluate(
            X_train.values, y_train, model,
            transformer=cv_transformers[fs_name],
            use_sample_weight=use_sw, verbose=True
        )
        cv_results[model_name][fs_name] = scores
        print(f'  → Accuracy : {scores["acc_mean"]:.4f} ± {scores["acc_std"]:.4f}')
        print(f'  → Macro F1 : {scores["f1_mean"]:.4f} ± {scores["f1_std"]:.4f}')

## Results Summary

In [ ]:
summary_rows = []
for model_name, fs_scores in cv_results.items():
    for fs_name, scores in fs_scores.items():
        summary_rows.append({
            'Model'       : model_name,
            'Feature Set' : fs_name,
            'N Features'  : feature_sets[fs_name]['n_features'],
            'CV Accuracy' : f"{scores['acc_mean']:.4f} ± {scores['acc_std']:.4f}",
            'CV Macro F1' : f"{scores['f1_mean']:.4f} ± {scores['f1_std']:.4f}",
        })

summary_df = pd.DataFrame(summary_rows)
print(summary_df.to_string(index=False))

In [ ]:
fs_labels = list(feature_sets.keys())
x     = np.arange(len(fs_labels))
width = 0.2
colors = {'Random Forest': ('steelblue', 'cornflowerblue'),
          'XGBoost'      : ('darkorange', 'moccasin')}

fig, ax = plt.subplots(figsize=(10, 5))
offsets = [-1.5, -0.5, 0.5, 1.5]
bars_meta = [
    ('Random Forest', 'acc', offsets[0], colors['Random Forest'][0], 'RF Accuracy'),
    ('Random Forest', 'f1',  offsets[1], colors['Random Forest'][1], 'RF Macro F1'),
    ('XGBoost',       'acc', offsets[2], colors['XGBoost'][0],       'XGB Accuracy'),
    ('XGBoost',       'f1',  offsets[3], colors['XGBoost'][1],       'XGB Macro F1'),
]

for model_name, metric, offset, color, label in bars_meta:
    means = [cv_results[model_name][fs][f'{metric}_mean'] for fs in fs_labels]
    stds  = [cv_results[model_name][fs][f'{metric}_std']  for fs in fs_labels]
    ax.bar(x + offset * width, means, width, yerr=stds, capsize=3,
           label=label, color=color, alpha=0.85)

ax.set_xticks(x)
ax.set_xticklabels([f.upper() for f in fs_labels])
ax.set_ylabel('Score')
ax.set_title('Random Forest & XGBoost — 5-Fold CV by Feature Space')
ax.set_ylim(0, 1.05)
ax.legend(fontsize=8)
plt.tight_layout()
plt.savefig('../figures/06-Forests/forests_cv_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

## Save Results

In [ ]:
joblib.dump(cv_results, '../data/processed/cv_results_forests.pkl')

# Re-train on full training set for use in notebook 08
trained_models = {}
for model_name, (model, use_sw) in models.items():
    trained_models[model_name] = {}
    for fs_name, fs in feature_sets.items():
        m = clone(model)
        fit_kwargs = {}
        if use_sw:
            fit_kwargs['sample_weight'] = compute_sample_weight('balanced', y_train)
        m.fit(fs['X_train'], y_train.values, **fit_kwargs)
        trained_models[model_name][fs_name] = m
        print(f'Trained {model_name} ({fs_name})')

joblib.dump(trained_models, '../data/processed/trained_models_forests.pkl')
print('\nResults and models saved to data/processed/')